# Lesson 2B: Hierarchical Clustering - Practical

## From Theory to Production

In Lesson 2A, we learned the mathematics of hierarchical clustering and built it from scratch. Now let's use **production-quality libraries** to solve real-world problems!

## Today's Case Study: Gene Expression Analysis 🧬

You're a bioinformatics researcher at a cancer research institute. You have gene expression data from 100 patients with different cancer types. **Your goal**: Find groups of patients with similar gene expression patterns to:

1. **Identify cancer subtypes** not visible with traditional diagnostics
2. **Personalize treatment** - different subtypes need different therapies  
3. **Predict outcomes** - some subtypes are more aggressive
4. **Discover biomarkers** - which genes define each subtype?

This is a real use case! Hierarchical clustering on gene expression led to discoveries like:
- **Breast cancer subtypes**: Luminal A, Luminal B, HER2+, Triple-negative
- **Leukemia classification**: ALL vs AML subtypes
- **Drug response prediction**: Which patients respond to which drugs

Let's build a complete pipeline using **scikit-learn** and **scipy**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score, davies_bouldin_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded successfully!")

## Step 1: Generate Synthetic Gene Expression Data

For this tutorial, we'll simulate gene expression data that mimics real cancer datasets.

**Real datasets** like TCGA (The Cancer Genome Atlas) have:
- Rows = patients (samples)
- Columns = genes (features, often 20,000+)
- Values = expression levels (how much each gene is expressed)

We'll create a simplified version with 3 cancer subtypes.

In [ ]:
# Create synthetic "gene expression" data
print("Generating synthetic gene expression dataset...")
print("="*70)

# 100 patients, 50 genes, 3 cancer subtypes
n_patients = 100
n_genes = 50
n_subtypes = 3

X, y_true = make_blobs(
    n_samples=n_patients,
    n_features=n_genes,
    centers=n_subtypes,
    cluster_std=1.5,
    random_state=42
)

# Create DataFrame with gene names
gene_names = [f'Gene_{i+1}' for i in range(n_genes)]
patient_ids = [f'Patient_{i+1:03d}' for i in range(n_patients)]

df = pd.DataFrame(X, columns=gene_names, index=patient_ids)

print(f"Dataset shape: {df.shape}")
print(f"  • {df.shape[0]} patients (samples)")
print(f"  • {df.shape[1]} genes (features)")
print(f"  • {n_subtypes} true cancer subtypes (hidden from algorithm)")
print(f"\nFirst 5 patients, first 5 genes:")
print(df.iloc[:5, :5])
print("\n✅ Dataset created!")

## Step 2: Preprocessing

**Critical for biological data!**

Gene expression data often has:
- **Different scales**: Some genes have high expression, others low
- **Different variances**: Some genes vary a lot, others are stable
- **Skewed distributions**: Often log-transformed in practice

**Solution**: Standardize features (z-score normalization)

$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
# Standardize gene expression levels
print("Preprocessing: Standardizing gene expression...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Create scaled DataFrame
df_scaled = pd.DataFrame(X_scaled, columns=gene_names, index=patient_ids)

# Compare before/after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before scaling
axes[0].hist(X.ravel(), bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_title('Before Scaling', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Expression Level', fontweight='bold')
axes[0].set_ylabel('Frequency', fontweight='bold')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].grid(True, alpha=0.3)

# After scaling
axes[1].hist(X_scaled.ravel(), bins=50, alpha=0.7, color='green', edgecolor='black')
axes[1].set_title('After Scaling (Standardized)', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Standardized Expression', fontweight='bold')
axes[1].set_ylabel('Frequency', fontweight='bold')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Effect of Standardization on Gene Expression', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

print("✅ Data standardized!")
print(f"   Mean ≈ {X_scaled.mean():.2e} (should be ~0)")
print(f"   Std ≈ {X_scaled.std():.2f} (should be ~1)")

## Step 3: Hierarchical Clustering with Scipy

**Scipy** provides the classic interface for hierarchical clustering via `linkage()`.

**Workflow:**
1. `linkage()` → compute hierarchical clustering
2. `dendrogram()` → visualize the tree
3. `fcluster()` → cut tree to get cluster labels

In [ ]:
# Perform hierarchical clustering with different linkages
print("Performing hierarchical clustering with different linkage methods...")
print("="*70)

linkage_methods = {
    'ward': 'Ward (minimize variance)',
    'average': 'Average (UPGMA)',
    'complete': 'Complete (MAX)',
    'single': 'Single (MIN)'
}

linkage_results = {}

for method_key, method_name in linkage_methods.items():
    print(f"\n{method_name}...")
    Z = linkage(X_scaled, method=method_key)
    linkage_results[method_key] = Z
    print(f"  ✅ Computed {len(Z)} merges")

print("\n" + "="*70)
print("✅ All linkage methods completed!")

In [ ]:
# Create comprehensive dendrogram comparison
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.ravel()

colors = ['purple', 'green', 'blue', 'red']

for idx, (method_key, Z) in enumerate(linkage_results.items()):
    ax = axes[idx]
    
    # Create dendrogram
    dendro = dendrogram(
        Z,
        ax=ax,
        truncate_mode='lastp',  # Show only last p merges
        p=30,  # Show last 30 merges for readability
        leaf_font_size=8,
        color_threshold=0,
        above_threshold_color=colors[idx]
    )
    
    method_name = linkage_methods[method_key]
    ax.set_title(f'{method_name} Linkage', fontweight='bold', fontsize=14)
    ax.set_xlabel('Patient (or merged cluster)', fontweight='bold')
    ax.set_ylabel('Distance', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add horizontal line for 3-cluster cut
    if method_key == 'ward':
        threshold = Z[-2, 2] + (Z[-1, 2] - Z[-2, 2]) / 2
    else:
        threshold = (Z[-3, 2] + Z[-2, 2]) / 2
    ax.axhline(y=threshold, color='black', linestyle='--', linewidth=2.5,
               label=f'3-cluster cut (y={threshold:.2f})', alpha=0.7)
    ax.legend(fontsize=10)

plt.suptitle('Hierarchical Clustering: Dendrogram Comparison\n(Gene Expression Data)',
             fontweight='bold', fontsize=16, y=0.995)
plt.tight_layout()
plt.show()

print("✅ Dendrograms created!")
print("\nKey Observations:")
print("• Ward: Balanced tree, clear 3-group structure")
print("• Average: Similar to Ward, slightly different heights")
print("• Complete: Compact merges, clear separation")
print("• Single: May show chaining (long branches)")

## Step 4: Extract Clusters and Evaluate

Now let's cut the dendrogram to get 3 clusters (matching our known subtypes) and evaluate quality.

In [ ]:
# Extract 3 clusters from each linkage method
print("Extracting 3 clusters from each linkage method...")
print("="*70)

n_clusters = 3
cluster_labels = {}

for method_key, Z in linkage_results.items():
    labels = fcluster(Z, n_clusters, criterion='maxclust')
    cluster_labels[method_key] = labels
    
    # Compute silhouette score
    silhouette = silhouette_score(X_scaled, labels)
    db_score = davies_bouldin_score(X_scaled, labels)
    
    method_name = linkage_methods[method_key]
    print(f"\n{method_name}:")
    print(f"  Silhouette Score: {silhouette:.4f} (higher is better, max=1)")
    print(f"  Davies-Bouldin:   {db_score:.4f} (lower is better, min=0)")
    print(f"  Cluster sizes: {np.bincount(labels)}")

print("\n" + "="*70)
print("✅ Clusters extracted and evaluated!")

In [ ]:
# Visualize clustering results (using PCA for 2D projection)
from sklearn.decomposition import PCA

# Project to 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")

# Plot clustering results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: True labels
ax = axes[0, 0]
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_true, cmap='Set1',
                     s=100, alpha=0.6, edgecolors='k', linewidth=1)
ax.set_title('Ground Truth\n(3 Cancer Subtypes)', fontweight='bold', fontsize=12)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)', fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='True Subtype')

# Plot clustering results for each linkage
for idx, (method_key, labels) in enumerate(cluster_labels.items(), start=1):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='viridis',
                        s=100, alpha=0.6, edgecolors='k', linewidth=1)
    
    silhouette = silhouette_score(X_scaled, labels)
    method_name = linkage_methods[method_key]
    ax.set_title(f'{method_name}\nSilhouette: {silhouette:.3f}',
                fontweight='bold', fontsize=12)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)', fontweight='bold')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)', fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='Cluster')

# Hide extra subplot
axes[1, 2].axis('off')

plt.suptitle('Hierarchical Clustering Results: Gene Expression Data\n(Projected to 2D via PCA)',
             fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

print("✅ Clustering results visualized!")

## Step 5: Scikit-Learn Interface

**Scikit-learn** provides a more modern, sklearn-style API with `AgglomerativeClustering`.

**Advantages:**
- Fits sklearn conventions (`fit()`, `fit_predict()`)
- Works with sklearn pipelines
- Can specify connectivity constraints
- Supports distance thresholds instead of n_clusters

In [ ]:
# Use sklearn's AgglomerativeClustering
print("Using scikit-learn's AgglomerativeClustering...")
print("="*70)

# Ward linkage with 3 clusters
hc_ward = AgglomerativeClustering(
    n_clusters=3,
    linkage='ward',
    metric='euclidean'  # Ward requires Euclidean
)

labels_ward = hc_ward.fit_predict(X_scaled)

print(f"\n✅ Ward clustering completed!")
print(f"   Cluster sizes: {np.bincount(labels_ward)}")
print(f"   Number of leaves: {hc_ward.n_leaves_}")
print(f"   Number of connected components: {hc_ward.n_connected_components_}")

# Evaluate
silhouette = silhouette_score(X_scaled, labels_ward)
db_score = davies_bouldin_score(X_scaled, labels_ward)

print(f"\n   Silhouette Score: {silhouette:.4f}")
print(f"   Davies-Bouldin: {db_score:.4f}")

print("\n" + "="*70)

## Step 6: Finding Optimal Number of Clusters

Unlike K-Means, we can explore the full dendrogram to choose k!

**Two approaches:**
1. **Elbow method** on dendrogram heights
2. **Silhouette analysis** across different k

In [ ]:
# Method 1: Elbow method on linkage distances
Z_ward = linkage_results['ward']

# Last 10 merge distances
last_merges = Z_ward[-10:, 2]
merge_numbers = range(len(Z_ward) - 10, len(Z_ward))

plt.figure(figsize=(14, 6))

# Plot 1: Last merge distances
plt.subplot(1, 2, 1)
plt.plot(merge_numbers, last_merges, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Merge Number', fontweight='bold')
plt.ylabel('Merge Distance', fontweight='bold')
plt.title('Last 10 Merge Distances (Ward Linkage)', fontweight='bold', fontsize=14)
plt.grid(True, alpha=0.3)

# The jump between merges indicates good k
# Larger jump = better separation
for i in range(len(last_merges) - 1):
    diff = last_merges[i+1] - last_merges[i]
    plt.text(merge_numbers[i] + 0.5, last_merges[i], f'+{diff:.2f}',
            fontsize=8, ha='center')

# Plot 2: Silhouette score vs k
plt.subplot(1, 2, 2)

k_range = range(2, 11)
silhouette_scores = []

for k in k_range:
    labels = fcluster(Z_ward, k, criterion='maxclust')
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)', fontweight='bold')
plt.ylabel('Silhouette Score', fontweight='bold')
plt.title('Silhouette Score vs Number of Clusters', fontweight='bold', fontsize=14)
plt.axvline(x=3, color='red', linestyle='--', linewidth=2, label='True k=3')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

# Find optimal k
optimal_k = k_range[np.argmax(silhouette_scores)]
print(f"✅ Optimal number of clusters: k={optimal_k}")
print(f"   Silhouette score: {max(silhouette_scores):.4f}")
print(f"   (True k=3 for our synthetic data)")

## Step 7: Identifying Key Genes (Biomarkers)

Once we have clusters, we can identify which genes differentiate the subtypes!

**Method**: For each gene, compute ANOVA F-statistic
- High F-statistic = gene expression differs significantly across clusters
- These are potential **biomarkers** for cancer subtypes

In [ ]:
from scipy.stats import f_oneway

# Use Ward clustering labels
labels = cluster_labels['ward']

print("Identifying differentially expressed genes (biomarkers)...")
print("="*70)

# For each gene, test if expression differs across clusters
gene_scores = []

for gene_idx in range(n_genes):
    gene_expression = X_scaled[:, gene_idx]
    
    # Split by cluster
    clusters = [gene_expression[labels == (i+1)] for i in range(n_clusters)]
    
    # ANOVA F-test
    f_stat, p_value = f_oneway(*clusters)
    
    gene_scores.append({
        'Gene': gene_names[gene_idx],
        'F-statistic': f_stat,
        'p-value': p_value
    })

# Convert to DataFrame and sort
biomarkers_df = pd.DataFrame(gene_scores)
biomarkers_df = biomarkers_df.sort_values('F-statistic', ascending=False)

print(f"\nTop 10 Biomarker Genes (highest F-statistic):")
print(biomarkers_df.head(10).to_string(index=False))

# Visualize top biomarkers
top_genes = biomarkers_df.head(6)['Gene'].values

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, gene in enumerate(top_genes):
    ax = axes[idx]
    gene_idx = gene_names.index(gene)
    gene_expr = X_scaled[:, gene_idx]
    
    # Boxplot by cluster
    data_by_cluster = [gene_expr[labels == (i+1)] for i in range(n_clusters)]
    bp = ax.boxplot(data_by_cluster, labels=[f'Subtype {i+1}' for i in range(n_clusters)],
                    patch_artist=True)
    
    # Color boxes
    colors = ['lightblue', 'lightgreen', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    f_stat = biomarkers_df[biomarkers_df['Gene'] == gene]['F-statistic'].values[0]
    ax.set_title(f'{gene}\nF-stat={f_stat:.2f}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Standardized Expression', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Top 6 Biomarker Genes: Expression Across Cancer Subtypes',
             fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

print("\n✅ Biomarker analysis complete!")
print("These genes show significantly different expression across subtypes.")
print("In real research, these would be validated experimentally!")

## Summary & Best Practices

### 🎯 What We Learned

1. **Production Tools**:
   - `scipy.cluster.hierarchy.linkage()` for clustering
   - `scipy.cluster.hierarchy.dendrogram()` for visualization
   - `sklearn.cluster.AgglomerativeClustering` for sklearn-style API

2. **Complete Pipeline**:
   - Preprocessing (standardization is critical!)
   - Clustering with multiple linkages
   - Evaluation (silhouette, Davies-Bouldin)
   - Optimal k selection
   - Biomarker discovery

3. **Real Application**: Gene expression clustering for cancer subtype discovery

### 🔬 Best Practices

✅ **Always preprocess** biological data (standardize or normalize)  
✅ **Try multiple linkages** - Ward is usually best, but not always  
✅ **Use silhouette score** to validate and choose k  
✅ **Visualize** with dendrograms and PCA projections  
✅ **Interpret results** - find biomarkers or key features  

### 🚀 Next Steps

**In production**, you would:
1. **Use real data**: TCGA, GEO datasets (10,000+ genes)
2. **Filter genes**: Remove low-variance genes, keep top 1000-5000
3. **Dimensionality reduction**: PCA before clustering for speed
4. **Validate**: Use clinical outcomes to verify subtypes
5. **Cross-validate**: Check stability across different datasets

**Related Methods**:
- **DBSCAN** (Lesson 3) for arbitrary shapes and noise
- **GMM** (Lesson 4) for soft clustering
- **PCA** (Lesson 5) to reduce dimensions first

---

**Key Takeaway**: Hierarchical clustering + gene expression = discovered cancer subtypes that changed medicine! This same approach works for any domain with hierarchical structure: customers, documents, species, proteins, etc.